Librerías necesarias 

In [1]:
import pandas as pd
import networkx as nx
import json
import re
from pprint import pprint

import overpy
import folium
import os
import math
import numpy as np
from pathlib import Path
from rapidfuzz import process, fuzz


Ecuaciones de distancias

In [2]:
def haversine_m(lat1, lon1, lat2_arr, lon2_arr):
    R = 6371000.0
    p1 = np.radians(lat1); p2 = np.radians(lat2_arr)
    dlat = p2 - p1
    dlon = np.radians(lon2_arr) - np.radians(lon1)
    a = np.sin(dlat/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlon/2)**2
    c = 2*np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R*c

def numerable(df, cols):
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


# Base de datos

### GTFS Ciudad de México (tabular)
General Transit Feed Specification Ciudad de México

### DENUE INEGI (texto)
Contiene  la descripcion de la razón comercial por tipo de establecimiento  de la Ciudad de México
Los datos están filtrados por el codigo de sector de salud (62)
Especifica las caracteristicas y ramas de cada categoria, que especialidades atiende y sus posibles variables:
- acceso: publico o privado
- Creacion de diccionario de de especialidades medicas para las consultas utilizando la clasificacion SCIAN: Sistema de Clasificación Industrial de América del Norte

### Mapa de locaciones (grafo)
- Limpieza de datos de ubicacion de hospitales, considerando una distancia de 100 m para el cluster de los mismos, conservando sus datos originales, es decir, generando un conjunto de locaciones.
- Limpieza de datos de paradas, considerando una distancia de 50 m para la union de las mismas.


### Limpieza de la base de datos de texto
Creacion de diccionario para futuras consultas. La limpieza incluye lo siguiente:
- normalizacion de formato: minusculas, correccion de caracteres especiales
- asegurarse de que el texto tiene un identificador unico

In [3]:
BD_text = 'SCIAN.xlsx'
df = pd.read_excel(BD_text, header=None)

codigo = None
sector = None
dictionary = {}

filtros = ["blanqueamiento dental", "cirugia dental", "consulta dental",
           "endodoncia", "extraccion dental", "odontologia",
           "ortodoncia", "periodoncia", "prótesis dental", "maxilofacial",
           "consultorios dentales", "clinica dental", "medicos internistas",
              "medicina interna", "medico internista", "medicos generales",
              "consulta medica externa", "consultorio medico", "clinica medica",
              "consultorios de medicina general", "atención médica para el control de la natalidad",
                "atención médica para el control de la salud reproductiva", "atención psicoprofiláctica",
                "atención prenatal", "atención postnatal", "atención médica para la planificación familiar",
                "atención médica para la salud reproductiva", "atención médica para la salud sexual",
                "atención médica para la salud materna", "atención médica para la salud infantil",
           ]


for index, row in df.iterrows():
    num_codigo = str(row[0]).strip()
    titulo = str(row[1]).strip().lower() if pd.notna(row[1]) else ''
    servicio = str(row[2]).strip().lower() if pd.notna(row[2]) else ''
    
    if num_codigo.isdigit() and len(num_codigo) == 6:
        codigo = num_codigo
        if 'privado' in titulo and 'público' in titulo:
            sector = 'ambos'
        elif 'privado' in titulo:
            sector = 'privado'
        elif 'público' in titulo:
            sector = 'público'
        else:
            sector = 'público'  
        continue
    
    # descripciones del codigo asociado al servicio

    if servicio and codigo:
        #parts = re.split(r'\s+en\s+|,\s*', servicio, maxsplit=1)  
        #parts = re.split(r'\s+en\s+|,\s*|\s+del\s+', servicio, maxsplit=1) 
        parts = re.split(r'\s+en\s+|,\s*|\s+del\s+|\s+por el\s+', servicio, maxsplit=1) 
        key = parts[0].strip() 
        if key and key not in dictionary: 
            dictionary[key] = {
                 'codigo': codigo, 
                 'sector': sector, 
                 }

   

salida = 'diccionario.json'
with open(salida, 'w', encoding='utf-8') as f:
    json.dump(dictionary, f, ensure_ascii=False, indent=2)

print(f"Diccionario con {len(dictionary)} especialidades, '{salida}'")


Diccionario con 250 especialidades, 'diccionario.json'


In [4]:
from collections import defaultdict

BD_text = 'SCIAN.xlsx'
df = pd.read_excel(BD_text, header=None)

codigo = None
sector = None
diccionario = defaultdict(lambda: {"codigo": [], "sector": []})

# Lista de filtros opcional
filtros = ["blanqueamiento dental", "cirugia dental", "consulta dental",
           "endodoncia", "extraccion dental", "odontologia",
           "ortodoncia", "periodoncia", "prótesis dental", "maxilofacial",
           "consultorios dentales", "clinica dental", "medicos internistas",
           "medicina interna", "medico internista", "medicos generales",
           "consulta medica externa", "consultorio medico", "clinica medica",
           "consultorios de medicina general", "atención médica para el control de la natalidad",
           "atención médica para el control de la salud reproductiva", "atención psicoprofiláctica",
           "atención prenatal", "atención postnatal", "atención médica para la planificación familiar",
           "atención médica para la salud reproductiva", "atención médica para la salud sexual",
           "atención médica para la salud materna", "atención médica para la salud infantil"]

for _, row in df.iterrows():
    num_codigo = str(row[0]).strip()
    titulo = str(row[1]).strip().lower() if pd.notna(row[1]) else ''
    servicio = str(row[2]).strip().lower() if pd.notna(row[2]) else ''

    
    if num_codigo.isdigit() and len(num_codigo) == 6:
        codigo = num_codigo
        if 'privado' in titulo and 'público' in titulo:
            sector = ['privado', 'público']
        elif 'privado' in titulo:
            sector = ['privado']
        elif 'público' in titulo:
            sector = ['público']
        else:
            sector = ['público'] 
        continue

    if servicio and codigo:
        parts = re.split(r'\s+en\s+|,\s*|\s+del\s+|\s+por el\s+', servicio, maxsplit=1)
        key = parts[0].strip()

        for sector_actual in sector:
            if codigo not in diccionario[key]["codigo"]:
                diccionario[key]["codigo"].append(codigo)
            if sector_actual not in diccionario[key]["sector"]:
                diccionario[key]["sector"].append(sector_actual)

salida = 'diccionario_v2.json'
with open(salida, 'w', encoding='utf-8') as f:
    json.dump(diccionario, f, ensure_ascii=False, indent=2)

print(f"Diccionario con {len(diccionario)} especialidades, '{salida}'")


Diccionario con 250 especialidades, 'diccionario_v2.json'


Las especialidades se arreglan en un .json para evitar duplicados, sin embargo, se realiza una verificacion de duplicados para verificar posibles casos dados acentos o mayusculas.

In [5]:
with open("diccionario_v2.json", "r", encoding="utf-8") as f:
    diccionario = json.load(f)

especialidades_limpias = [clave.strip().lower() for clave in diccionario.keys()]
from collections import Counter
duplicados = {k: v for k, v in Counter(especialidades_limpias).items() if v > 1}

duplicados


{}

### Limpieza de la base de datos tabular

Columnas consideradas:
- nom_estab : Nombre del establecimiento, util para saber que es
- codigo_act : Codigo de actividad, del SCIAN, aca se hace el enlace con la BD de texto
- nombre_act : Descripcion de la actividad
- alcaldia : Util para la particion de dependencias
- latitud : Util para el analisis geoespacial
- altitud : Util para el analisis geoespacial 

Creamos un diccionario que incluya el codigo_act y el nombre_act, el cual nos servira de enlace para la BD de texto

In [6]:
import pandas as pd

df = pd.read_csv("INEGI_denue_62.csv")
df.columns = [col.strip().lower() for col in df.columns]

# ver las filas
dup = df[df.duplicated()]
print(f"\nFilas duplicadas completas: {len(dup)}")
if not dup.empty:
    print(dup.head())

# nombres
nombres = df[df.duplicated('nom_estab', keep=False)].sort_values('nom_estab')
print(f"\nDuplicados por nombre del establecimiento: {len(nombres)}")
if not nombres.empty:
    print(nombres.head())

# coordenadas
coords = df[df.duplicated(subset=['latitud', 'longitud'], keep=False)]
print(f"\nDuplicados por coordenadas: {len(coords)}")
if not coords.empty:
    print(coords.head())



Filas duplicadas completas: 1170
                       nom_estab  codigo_act  \
46        CENTRO MEDICO COYOACAN      621111   
56               CIRUGIA GENERAL      621111   
57               CIRUGIA GENERAL      621111   
140             CONSULTA GENERAL      621111   
184  CONSULTORIO CIRUGIA GENERAL      621111   

                                            nombre_act        alcaldia  \
46   Consultorios de medicina general del sector pr...        Coyoacán   
56   Consultorios de medicina general del sector pr...         Tlalpan   
57   Consultorios de medicina general del sector pr...         Tlalpan   
140  Consultorios de medicina general del sector pr...        Coyoacán   
184  Consultorios de medicina general del sector pr...  Miguel Hidalgo   

       latitud   longitud  
46   19.309760 -99.129702  
56   19.296174 -99.161182  
57   19.296174 -99.161182  
140  19.309169 -99.163286  
184  19.399727 -99.173042  

Duplicados por nombre del establecimiento: 9712
      nom_estab

In [7]:
folder = "duplicados_tab"
os.makedirs(folder, exist_ok=True)
dup.to_csv(f"{folder}/dup.csv", index=False)
nombres.to_csv(f"{folder}/nombres.csv", index=False)
coords.to_csv(f"{folder}/coords.csv", index=False)

In [8]:
df_clean = df[df['nom_estab'].str.strip().str.upper() != "SIN NOMBRE"]
df_clean = df_clean.drop_duplicates(subset=['latitud', 'longitud'], keep='first')

output_file = "DENUE_limpia.csv"
df_clean.to_csv(output_file, index=False, encoding='utf-8')

print(f"listo")

listo


In [9]:
df = pd.read_csv("DENUE_limpia.csv")
df.columns = [col.strip().lower() for col in df.columns]

# ver las filas
dup = df[df.duplicated()]
print(f"\nFilas duplicadas completas: {len(dup)}")
if not dup.empty:
    print(dup.head())

# nombres
nombres = df[df.duplicated('nom_estab', keep=False)].sort_values('nom_estab')
print(f"\nDuplicados por nombre del establecimiento: {len(nombres)}")
if not nombres.empty:
    print(nombres.head())

# coordenadas
coords = df[df.duplicated(subset=['latitud', 'longitud'], keep=False)]
print(f"\nDuplicados por coordenadas: {len(coords)}")
if not coords.empty:
    print(coords.head())



Filas duplicadas completas: 0

Duplicados por nombre del establecimiento: 6590
      nom_estab  codigo_act  \
15187        AA      624191   
15189        AA      624191   
15190        AA      624191   
14261        AA      623221   
14260        AA      623221   

                                              nombre_act  \
15187  Agrupaciones de autoayuda para alcohólicos y p...   
15189  Agrupaciones de autoayuda para alcohólicos y p...   
15190  Agrupaciones de autoayuda para alcohólicos y p...   
14261  Residencias del sector privado para el cuidado...   
14260  Residencias del sector privado para el cuidado...   

                     alcaldia    latitud   longitud  
15187                Coyoacán  19.331516 -99.160181  
15189     Venustiano Carranza  19.446025 -99.106112  
15190          Miguel Hidalgo  19.447861 -99.201081  
14261  La Magdalena Contreras  19.307184 -99.265384  
14260  La Magdalena Contreras  19.311877 -99.262461  

Duplicados por coordenadas: 0


In [10]:
import pandas as pd
import json

df = pd.read_csv('DENUE_limpia.csv')
df.columns = [col.strip().lower() for col in df.columns]

with open('diccionario_v2.json', 'r', encoding='utf-8') as f:
    text_dict = json.load(f)


cod_a_nom = {}
for term, data in text_dict.items():
    codigos = data.get('codigo')
    sector = data.get('sector')

    # Asegurar que codigos sea lista
    if not isinstance(codigos, list):
        codigos = [codigos]
    if not isinstance(sector, list):
        sector = [sector]

    for c, s in zip(codigos, sector):
        if c not in cod_a_nom:
            cod_a_nom[c] = s
        else:
            # Si ya existía y es diferente, lo marcas como mixto
            if cod_a_nom[c] != s:
                cod_a_nom[c] = "mixto"

df['sector_v1'] = df['codigo_act'].astype(str).map(lambda x: cod_a_nom.get(x, 'público'))

output_file = 'diccionario_v2.csv'
df.to_csv(output_file, index=False, encoding='utf-8')

print(f"listo")


listo


### Limpieza de la base de datos grafo

Columnas consideradas:
- stop_id
- stop_name
- stop_lat
- stop_lon
 
estas ultimas seran la conexion asociada a la base de datos tabular

In [11]:
import pandas as pd

BD_grafo = 'gtfs-2\stops.txt' 
df_paradas = pd.read_csv(BD_grafo)

df_paradas.columns = [col.strip().lower() for col in df_paradas.columns]

# definir las columnas 
df_paradas_limpias = df_paradas[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']]


df_paradas_limpias = df_paradas_limpias.dropna(subset=['stop_lat', 'stop_lon'])
df_paradas_limpias = df_paradas_limpias.drop_duplicates(subset=['stop_lat', 'stop_lon'])


salida = 'stops_limpios.csv'
df_paradas_limpias.to_csv(salida, index=False, encoding='utf-8')

print(f"listo")


<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:3: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8384\3733606279.py:3: SyntaxWarning: invalid escape sequence '\s'
  BD_grafo = 'gtfs-2\stops.txt'


listo


algunas paradas tienen coordenadas distintas pero estan a menos de 50m por lo tanto representan el mismo lugar fisico

Por lo que se agrupan las paradas por proximidad

In [12]:
from sklearn.neighbors import BallTree
from sklearn.metrics.pairwise import haversine_distances

df = pd.read_csv("stops_limpios.csv")
df.columns = [col.strip().lower() for col in df.columns]

#  el id de la agencia esta dado por los prim 2 de stop id
df['agencia'] = df['stop_id'].astype(str).str[:2]

coords_rad = np.radians(df[['stop_lat', 'stop_lon']].to_numpy())
df['coords_rad'] = list(coords_rad)  

# agrupar
grupo_id = 0
df['grupo_stop_id'] = -1  
asignados = set()

# parametros
radio_metros = 50
radio_radianes = radio_metros / 6371000.0  # tierra en m

for agencia, grupo_agencia in df.groupby('agencia'):
    coords = np.stack(grupo_agencia['coords_rad'].values)
    indices = grupo_agencia.index.to_list()

    tree = BallTree(coords, metric='haversine')
    vecinos = tree.query_radius(coords, r=radio_radianes)

    for i, lista_vecinos in enumerate(vecinos):
        i_global = indices[i]
        if i_global in asignados:
            continue
        grupo = [indices[j] for j in lista_vecinos if indices[j] not in asignados]
        if grupo:
            for idx in grupo:
                df.at[idx, 'grupo_stop_id'] = grupo_id
                asignados.add(idx)
            grupo_id += 1

# las equivalencias
equivalencias = df[['stop_id', 'stop_name', 'grupo_stop_id']].sort_values('grupo_stop_id')
equivalencias.to_csv("equivalencias_parada_id.csv", index=False, encoding='utf-8')

df.drop(columns=['coords_rad'], inplace=True)
df.to_csv("paradas_agrupadas.csv", index=False, encoding='utf-8')

print("listo mano")


KeyboardInterrupt: 

Ahora que se tienen tres fuentes de datos limpias y bien estructuradas — texto (servicios), tabular (establecimientos médicos), y paradas (transporte público) — se define la base de datos federada para hacer las consultas.

In [ ]:
# sistema para las consultas

# BD texto
with open('diccionario_v2.json', 'r', encoding='utf-8') as f:
    datos_json = json.load(f)

# Expandir codigos y sectores si son listas
servicios = []
for termino, info in datos_json.items():
    codigos = info['codigo'] if isinstance(info['codigo'], list) else [info['codigo']]
    sectores = info['sector'] if isinstance(info['sector'], list) else [info['sector']]
    
    for cod, sec in zip(codigos, sectores):
        servicios.append({
            'codigo': str(cod),
            'sector': sec,
            'termino': termino
        })

servicios = pd.DataFrame(servicios)

# BD tabular cds: centros de salud :D
cds = pd.read_csv("diccionario_v2.csv")
cds.columns = [col.strip().lower() for col in cds.columns]

# BD grafo
paradas = pd.read_csv("paradas_agrupadas.csv")
equivalencias = pd.read_csv("equivalencias_parada_id.csv")

paradas.columns = [col.strip().lower() for col in paradas.columns]
equivalencias.columns = [col.strip().lower() for col in equivalencias.columns]

# cds con tp
cds['codigo_act'] = cds['codigo_act'].astype(str)
servicios['codigo'] = servicios['codigo'].astype(str)

cds_full = cds.merge(
    servicios,
    how = "left",
    left_on="codigo_act",
    right_on= "codigo"
)

# paradas 
#paradas_full = paradas.merge(
#    equivalencias[['stop_id', 'grupo_stop_id']],
#    on="stop_id",
#    how="right"
#)

paradas_full = pd.read_csv("paradas_agrupadas.csv")
paradas_full.columns = [col.strip().lower() for col in paradas_full.columns]


#  cds con paradas cercanas (200 m)
coords_cds = np.radians(cds_full[['latitud', 'longitud']].to_numpy())
coords_par = np.radians(paradas_full[['stop_lat', 'stop_lon']].to_numpy())

tree = BallTree(coords_par, metric='haversine')
radio_m = 200
radio_r = radio_m / 6371000.0

indices = tree.query_radius(coords_cds, r=radio_r)


# vinculo cds paradas
vinculos = []
for idx_est, idxs_par in enumerate(indices):
    for idx_par in idxs_par:
        vinculos.append({
            "id_establecimiento": idx_est,
             "nombre_establecimiento": cds_full.iloc[idx_est]['nom_estab'],
            "codigo_actividad": cds_full.iloc[idx_est]['codigo_act'],
            "servicio": cds_full.iloc[idx_est]['nombre_act'],
            "sector": cds_full.iloc[idx_est]['sector_v1'],
            "stop_id": paradas_full.iloc[idx_par]['stop_id'],
            "nombre_parada": paradas_full.iloc[idx_par]['stop_name'],
            "grupo_stop_id": paradas_full.iloc[idx_par]['grupo_stop_id']
        })

df_vinculos = pd.DataFrame(vinculos)

df_vinculos.to_csv("vinculos_cds_tp.csv", index=False, encoding='utf-8')
print("liston")


liston


In [140]:
def metricas_calidad(establecimientos, paradas):
    total_est = len(establecimientos)
    total_par = len(paradas)

    print(" Métricas de calidad de la base federada\n")

    # completez
    print(" Completitud de columnas:")
    completitud_cols = establecimientos.notna().mean() * 100
    print(completitud_cols.round(2), "\n")

    # cds 
    con_servicio = establecimientos['nom_estab'].notna().sum()
    print(f" Servicios asignados: {con_servicio}/{total_est} ({con_servicio/total_est:.2%})")

    # % con coordenadas válidas
    coords_validas = establecimientos[['latitud', 'longitud']].notna().all(axis=1).sum()
    print(f"Coordenadas válidas: {coords_validas}/{total_est} ({coords_validas/total_est:.2%})")


    stop_dupes = paradas['stop_id'].duplicated().sum()
    print(f" IDs duplicados en paradas: {stop_dupes}")

# Ejecutar función
metricas_calidad(cds_full, paradas_full)


 Métricas de calidad de la base federada

 Completitud de columnas:
nom_estab     100.00
codigo_act    100.00
nombre_act    100.00
alcaldia      100.00
latitud       100.00
longitud      100.00
sector_v1     100.00
codigo         99.96
sector         99.96
termino        99.96
dtype: float64 

 Servicios asignados: 218804/218804 (100.00%)
Coordenadas válidas: 218804/218804 (100.00%)
 IDs duplicados en paradas: 0


### LLM 
Generacion de consultas heterogeneas con LLM
- Traducir  las 20 preguntas en lenguaje natural identificadas anteriormente y a código NetworkX, SQL y vectorial.
- El LLM debe reconocer que código especifico debe enviar a que base de datos.
- Se debe enviar el código y ejecutarse.
- Se debe reunir las respuestas y eliminar duplicados si es el caso.
- Se debe entregar el resultado al usuario.

In [13]:
# relacion de centros de salud
cds_full = pd.read_csv('diccionario_v2.csv')

# diccionario de especialidades
with open('diccionario_v2.json', 'r', encoding='utf-8') as f:
    diccionario = json.load(f)

# paradas de transporte público y vínculos
paradas_full = pd.read_csv('stops_limpios.csv')
vinculos = pd.read_csv('vinculos_cds_tp.csv')


cds_full['id_establecimiento'] = cds_full.index

conectividad = vinculos.groupby('id_establecimiento').agg(
    num_paradas = ('nombre_parada', 'nunique')
).reset_index()

conectividad = conectividad.merge(
    cds_full[['nom_estab', 'alcaldia', 'latitud', 'longitud']],
    left_on='id_establecimiento',
    right_index=True,
    how='left'
)

conectividad = conectividad.rename(columns={
    'nom_estab': 'nombre_establecimiento'
})

print("Registros sin alcaldía:", cds_full['alcaldia'].isna().sum())
print(conectividad.columns)

Registros sin alcaldía: 0
Index(['id_establecimiento', 'num_paradas', 'nombre_establecimiento',
       'alcaldia', 'latitud', 'longitud'],
      dtype='object')


In [142]:
def responder_pregunta(pregunta):
    pregunta = pregunta.lower()

    # buscar especialidad en el diccionario 
    servicio_clave = None
    for servicio in diccionario:
        if servicio in pregunta:
            servicio_clave = servicio
            break

    if not servicio_clave:
        print("No se encontró una especialidad válida")
        return

    # Extraer datos del diccionario
    codigos = diccionario[servicio_clave]['codigo']
    sectores = diccionario[servicio_clave]['sector']
    codigos = codigos if isinstance(codigos, list) else [codigos]
    sectores = sectores if isinstance(sectores, list) else [sectores]

    # si la pregunta incluye alcaldía
    alcaldias = cds_full['alcaldia'].dropna().unique()
    alcaldia_mencionada = next((a for a in alcaldias if a.lower() in pregunta), None)

    # filtro base, sobre el codigo
    filtro = cds_full['codigo_act'].astype(str).isin(codigos)

    # filtro por sector
    if 'público' in pregunta:
        filtro &= cds_full['sector_v1'].str.lower() == 'público'
    elif 'privado' in pregunta:
        filtro &= cds_full['sector_v1'].str.lower() == 'privado'
    else:
        filtro &= cds_full['sector_v1'].str.lower().isin([s.lower() for s in sectores])

    if alcaldia_mencionada:
        filtro &= cds_full['alcaldia'].str.lower() == alcaldia_mencionada.lower()

    resultados = cds_full[filtro]

    if resultados.empty:
        print("No se encontraron establecimientos con esa especialidad y filtros.")
        return

    print(f"\n Establecimientos encontrados: {len(resultados)}")
    print(resultados[['nom_estab', 'alcaldia', 'latitud', 'longitud']])

    # busqueda de paradas cercanas a los cds (centros de salud)
    ids_encontrados = resultados.index.tolist()
    nombres_estab_unicos = [
        f"{row['nom_estab']} ({idx})" for idx, row in resultados.iterrows()
    ]

    paradas_relacionadas = vinculos[vinculos['nombre_establecimiento'].isin(nombres_estab_unicos)]

    if not paradas_relacionadas.empty:
        print("\n Paradas cercanas:")
        print(paradas_relacionadas[['nombre_establecimiento', 'nombre_parada']].drop_duplicates())
    else:
        print("\nNo se encontraron paradas relacionadas para esos establecimientos.")

def conectados(n=5, alcaldia=None):

    data = conectividad.copy()
    if alcaldia:
        alcaldia = alcaldia.lower()
        data = data[data['alcaldia'].str.lower() == alcaldia]
        if data.empty:
            print(f"\n No se encontraron establecimientos en la alcaldía '{alcaldia.title()}'.")
            return

    print("\n Top {n} centros mejor conectados" + (f" en {alcaldia.title()}" if alcaldia else "") + ":")
    top_mejor = conectividad.sort_values(by='num_paradas', ascending=False).head(n)
    print(top_mejor[['nombre_establecimiento', 'alcaldia', 'num_paradas']])

    print("\n Top {n} centros peor conectados" + (f" en {alcaldia.title()}" if alcaldia else "") + ":")
    top_peor = conectividad[conectividad['num_paradas'] > 0].sort_values(by='num_paradas').head(n)
    print(top_peor[['nombre_establecimiento', 'alcaldia', 'num_paradas']])

    # Conectividad promedio por alcaldía
    alcaldias = conectividad.groupby('alcaldia').agg(
        promedio_paradas=('num_paradas', 'mean'),
        total_establecimientos=('nombre_establecimiento', 'count')
    ).reset_index().sort_values(by='promedio_paradas', ascending=False)

    print("\n Conectividad promedio por alcaldía:")
    print(alcaldias)

    # Conectividad por especialidad
    servicios = cds_full[['nom_estab', 'nombre_act']]
    conectividad_servicio = conectividad.merge(
        servicios,
        left_on='nombre_establecimiento',
        right_on='nom_estab',
        how='left'
    )

    especialidades = conectividad_servicio.groupby('nombre_act').agg(
        promedio_paradas=('num_paradas', 'mean'),
        total_establecimientos=('nombre_establecimiento', 'count')
    ).reset_index().sort_values(by='promedio_paradas', ascending=False)

    print("\nConectividad promedio por especialidad:")
    print(especialidades)




In [14]:
# Diccionario de preguntas descriptivas (ver las variantes de las mismas)
"¿Qué clínicas ofrecen cirugía maxilofacial?"
"¿Dónde hay análisis hormonales para personas en Coyoacán?"
"¿Cuáles son los consultorios dentales privados en la alcaldía Benito Juárez?"
"¿Qué establecimientos de oncología están en la alcaldía Cuauhtémoc?"
"¿Dónde hay clínicas dentales que realicen ortodoncia en la alcaldía Miguel Hidalgo?"
"Paradas cercanas a alergología privado en Tlalpan"
"top 3 conectados en Coyoacán"


'top 3 conectados en Coyoacán'

In [15]:
from sklearn.neighbors import BallTree


with open('diccionario_v2.json', 'r', encoding='utf-8') as f:
    datos_json = json.load(f)

servicios = []
for termino, info in datos_json.items():
    codigos = info['codigo'] if isinstance(info['codigo'], list) else [info['codigo']]
    sectores = info['sector'] if isinstance(info['sector'], list) else [info['sector']]
    
    for cod, sec in zip(codigos, sectores):
        servicios.append({
            'codigo': str(cod),
            'sector': sec,
            'termino': termino
        })

servicios = pd.DataFrame(servicios)

cds = pd.read_csv("diccionario_v2.csv")
cds.columns = [col.strip().lower() for col in cds.columns]
cds['codigo_act'] = cds['codigo_act'].astype(str)
cds['id_establecimiento'] = cds.index  # columna clave

servicios['codigo'] = servicios['codigo'].astype(str)
cds_full = cds.merge(servicios, how="left", left_on="codigo_act", right_on="codigo")

paradas_full = pd.read_csv("paradas_agrupadas.csv")
paradas_full.columns = [col.strip().lower() for col in paradas_full.columns]

coords_cds = np.radians(cds_full[['latitud', 'longitud']].to_numpy())
coords_par = np.radians(paradas_full[['stop_lat', 'stop_lon']].to_numpy())

tree = BallTree(coords_par, metric='haversine')
radio_r = 200 / 6371000.0  # 200 metros

indices = tree.query_radius(coords_cds, r=radio_r)

vinculos = []
for idx_est, idxs_par in enumerate(indices):
    for idx_par in idxs_par:
        vinculos.append({
            "id_establecimiento": cds_full.iloc[idx_est]['id_establecimiento'],
            "nombre_establecimiento": cds_full.iloc[idx_est]['nom_estab'],
            "codigo_actividad": cds_full.iloc[idx_est]['codigo_act'],
            "servicio": cds_full.iloc[idx_est]['termino'],
            "sector": cds_full.iloc[idx_est]['sector_v1'],
            "stop_id": paradas_full.iloc[idx_par]['stop_id'],
            "nombre_parada": paradas_full.iloc[idx_par]['stop_name'],
            "grupo_stop_id": paradas_full.iloc[idx_par]['grupo_stop_id']
        })

df_vinculos = pd.DataFrame(vinculos)
df_vinculos.to_csv("vinculos_cds_tp.csv", index=False, encoding='utf-8')
print("✔ Vinculación lista")


✔ Vinculación lista


In [16]:
conectividad = df_vinculos.groupby('id_establecimiento').agg(
    num_paradas=('nombre_parada', 'nunique')
).reset_index()

conectividad = conectividad.merge(
    cds_full[['id_establecimiento', 'nom_estab', 'alcaldia']],
    on='id_establecimiento',
    how='left'
).rename(columns={'nom_estab': 'nombre_establecimiento'})


In [29]:
def responder_pregunta(pregunta):
    pregunta = pregunta.lower()

    servicio_clave = next((s for s in datos_json if s in pregunta), None)
    if not servicio_clave:
        print("No se encontró una especialidad válida")
        return

    codigos = datos_json[servicio_clave]['codigo']
    sectores = datos_json[servicio_clave]['sector']
    codigos = codigos if isinstance(codigos, list) else [codigos]
    sectores = sectores if isinstance(sectores, list) else [sectores]

    alcaldias = cds_full['alcaldia'].dropna().unique()
    alcaldia_mencionada = next((a for a in alcaldias if a.lower() in pregunta), None)

    filtro = cds_full['codigo_act'].astype(str).isin(codigos)

    if 'público' in pregunta:
        filtro &= cds_full['sector_v1'].str.lower() == 'público'
    elif 'privado' in pregunta:
        filtro &= cds_full['sector_v1'].str.lower() == 'privado'
    else:
        filtro &= cds_full['sector_v1'].str.lower().isin([s.lower() for s in sectores])

    if alcaldia_mencionada:
        filtro &= cds_full['alcaldia'].str.lower() == alcaldia_mencionada.lower()

    resultados = cds_full[filtro]
    if resultados.empty:
        print("No se encontraron establecimientos.")
        return

    print(f"\nEstablecimientos encontrados: {len(resultados)}")
    print(resultados[['nom_estab', 'alcaldia', 'latitud', 'longitud']])

    vinculados = df_vinculos[df_vinculos['id_establecimiento'].isin(resultados['id_establecimiento'])]
    if not vinculados.empty:
        print("\nParadas cercanas:")
        print(vinculados[['nombre_establecimiento', 'nombre_parada']].drop_duplicates())
    else:
        print("\nNo se encontraron paradas cercanas.")

def conectados(n=5, alcaldia=None):
    data = conectividad.copy()
    if alcaldia:
        alcaldia = alcaldia.lower()
        data = data[data['alcaldia'].str.lower() == alcaldia]
        if data.empty:
            print(f"\n No se encontraron establecimientos en la alcaldía '{alcaldia.title()}'.")
            return

    print("\nTop {n} centros mejor conectados" + (f" en {alcaldia.title()}" if alcaldia else "") + ":")
    print(data.sort_values(by='num_paradas', ascending=False).head(n))

    print("\nTop {n} centros peor conectados" + (f" en {alcaldia.title()}" if alcaldia else "") + ":")
    print(data[data['num_paradas'] > 0].sort_values(by='num_paradas').head(n))

    alcaldias = data.groupby('alcaldia').agg(
        promedio_paradas=('num_paradas', 'mean'),
        total_establecimientos=('nombre_establecimiento', 'count')
    ).reset_index().sort_values(by='promedio_paradas', ascending=False)

    print("\nConectividad promedio por alcaldía:")
    print(alcaldias)

def prediccion_conectividad(lat, lon, radio_m=200):
    coord_rad = np.radians([[lat, lon]])
    tree = BallTree(np.radians(paradas_full[['stop_lat', 'stop_lon']].to_numpy()), metric='haversine')
    radio_r = radio_m / 6371000.0

    indices = tree.query_radius(coord_rad, r=radio_r)[0]
    num_paradas = len(indices)

    print(f"Número estimado de paradas dentro de {radio_m} m: {num_paradas}")

In [ ]:
# Diccionario de preguntas descriptivas (ver las variantes de las mismas)
"¿Qué clínicas ofrecen cirugía maxilofacial?"
"¿Dónde hay análisis hormonales para personas en Coyoacán?"
"¿Cuáles son los consultorios dentales privados en la alcaldía Benito Juárez?"
"¿Qué establecimientos de oncología están en la alcaldía Cuauhtémoc?"
"¿Dónde hay clínicas dentales que realicen ortodoncia en la alcaldía Miguel Hidalgo?"
"Paradas cercanas a alergología privado en Tlalpan"
"top 3 conectados en Coyoacán"


In [32]:
entrada = input("Haz tu pregunta: ").lower()

import re
match = re.search(r'top\s*(\d+)', entrada)
alcaldia_match = next((a for a in cds_full['alcaldia'].dropna().unique() if a.lower() in entrada), None)

if "conectado" in entrada or "conectividad" in entrada or "conexiones" in entrada or "conexión" in entrada or "conexion" in entrada or "conectados" in entrada:
    n = int(match.group(1)) if match else 5
    conectados(n, alcaldia=alcaldia_match)

if "predicción" in entrada or "prediccion" in entrada or "estimación" in entrada or "estimacion" in entrada or "predice" in entrada:
    prediccion_conectividad(19.4326, -99.1332, radio_m=200) 
else:
    responder_pregunta(entrada)



Establecimientos encontrados: 4736
                                               nom_estab    alcaldia  \
4807                     ALLERMEDICA MULTIESPECIALIDADES  Cuauhtémoc   
4808                     ALLERMEDICA MULTIESPECIALIDADES  Cuauhtémoc   
4809                     ALLERMEDICA MULTIESPECIALIDADES  Cuauhtémoc   
4810                     ALLERMEDICA MULTIESPECIALIDADES  Cuauhtémoc   
4811                     ALLERMEDICA MULTIESPECIALIDADES  Cuauhtémoc   
...                                                  ...         ...   
37103  CLINICA DE ESPECIALIDADES NUMERO 6 SERVICIO DE...  Cuauhtémoc   
37104  CLINICA DE ESPECIALIDADES NUMERO 6 SERVICIO DE...  Cuauhtémoc   
37105  CLINICA DE ESPECIALIDADES NUMERO 6 SERVICIO DE...  Cuauhtémoc   
37106  CLINICA DE ESPECIALIDADES NUMERO 6 SERVICIO DE...  Cuauhtémoc   
37107  CLINICA DE ESPECIALIDADES NUMERO 6 SERVICIO DE...  Cuauhtémoc   

         latitud   longitud  
4807   19.450551 -99.156354  
4808   19.450551 -99.156354  
4809   19

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor  # o clasificador
from sklearn.metrics import mean_squared_error, accuracy_score

# 1. Preparar datos
# Asumimos que ya tienes `conectividad` DataFrame con columnas:
# id_establecimiento, num_paradas, alcaldia, etc.
df_pred = conectividad.copy()

# Añadir variables adicionales (features)
# Ejemplo: sector_v1 (público / privado) desde cds_full
cds_full_small = cds_full[['id_establecimiento', 'sector_v1', 'codigo_act']]
df_pred = df_pred.merge(cds_full_small, on='id_establecimiento', how='left')

# Convertimos variables categóricas en dummy
df_pred = pd.get_dummies(df_pred, columns=['sector_v1'], drop_first=True)

# Definir target: por ejemplo, num_paradas (o bien categorizar en alto/bajo)
y = df_pred['num_paradas']
X = df_pred.drop(columns=['num_paradas', 'nombre_establecimiento', 'alcaldia'])

# 2. Dividir datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Entrenar modelo
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Evaluar
y_pred = model.predict(X_test)
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

# 5. Interpretación básica de importancia de variables
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Importancia de features:")
print(importances.head(10))
